# Instructor Effectiveness Modeling

**Role context:** Data Science / AI Content Specialist Intern
**Unit of raw data:** one course batch
**Question:** which instructors are more effective, and can engagement and feedback anticipate that?

This notebook uses only `dataset.csv` and the allowed stack (pandas, numpy, scikit-learn, matplotlib, seaborn). No external data and no AutoML.

Accuracy is not the goal. The goal is a definition of effectiveness that a product team could defend, an aggregation from batches to instructors that matches how the data is generated, and a model that is evaluated against a real baseline with its failure modes written down.


## How the analysis is set up

Each row is a batch. Instructors teach many batches and many courses, so a batch score is not an instructor score.

Two choices drive everything else:

1. **The label is built only from learner outcomes** (finishing, learning gain, quiz mastery). Engagement and feedback are *not* mixed into the label.
2. **The model is only allowed to use engagement, feedback, consistency, and teaching load.** It is not allowed to see the outcome columns that define the label.

If the same columns were used to build the label and to predict it, a linear model would mostly rediscover the formula. That can look like 95%+ accuracy and still teach the business nothing. A short comparison later in the notebook shows that contrast on purpose.

Tiers are **relative** (bottom, middle, and top third of this instructor pool). They are not an absolute pass mark.


## 0. Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path("dataset.csv")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.float_format", lambda v: f"{v:0.3f}")
np.random.seed(RANDOM_STATE)


## 1. Exploratory data analysis

### 1.1 Load and sanity checks

Expected shape from the brief: one row per batch, with identifiers (`batch_id`, `instructor_id`, `course_id`) plus outcome, engagement, and feedback measures.


In [ ]:
df = pd.read_csv(DATA_PATH)

ID_COLS = ["batch_id", "instructor_id", "course_id"]
OUTCOME_COLS = [
    "completion_rate",
    "dropout_rate",
    "avg_score_improvement",
    "avg_quiz_score",
]
ENGAGEMENT_COLS = [
    "avg_watch_time",
    "assignment_submission_rate",
    "forum_activity_rate",
]
FEEDBACK_COLS = ["avg_feedback_score", "feedback_response_rate"]
METRIC_COLS = OUTCOME_COLS + ENGAGEMENT_COLS + FEEDBACK_COLS

print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
print(f"Batches: {df['batch_id'].nunique():,}")
print(f"Instructors: {df['instructor_id'].nunique():,}")
print(f"Courses: {df['course_id'].nunique():,}")
print(f"Duplicate batch ids: {df['batch_id'].duplicated().sum()}")
print(f"Missing values: {int(df.isna().sum().sum())}")
display(df.head())
display(df[METRIC_COLS].describe().T)


### 1.2 Distributions

Looking for skew, piles of values at a floor or ceiling, and anything that should not be averaged naively.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, col in zip(axes.ravel(), METRIC_COLS):
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color="#3d5a80")
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("")
fig.suptitle("Batch-level distributions", y=1.01, fontsize=14)
fig.tight_layout()
plt.show()

# Values sitting exactly on the observed min or max are a censoring clue.
bound_rows = []
for col in METRIC_COLS:
    lo, hi = df[col].min(), df[col].max()
    bound_rows.append(
        {
            "metric": col,
            "min": lo,
            "max": hi,
            "n_at_min": int((df[col] == lo).sum()),
            "n_at_max": int((df[col] == hi).sum()),
        }
    )
bounds = pd.DataFrame(bound_rows)
display(bounds)


### 1.3 Relationships

The main risk at this stage is double-counting one fact under two column names, and mistaking a weak batch-level correlation for "no instructor effect."


In [ ]:
corr = df[METRIC_COLS].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    annot_kws={"size": 8},
)
ax.set_title("Batch-level correlations")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.2, 5.2))
sns.scatterplot(
    data=df,
    x="completion_rate",
    y="dropout_rate",
    s=12,
    alpha=0.35,
    color="#3d5a80",
    ax=ax,
)
ax.plot([0.2, 1.0], [0.8, 0.0], color="#c44536", lw=1.2, label="completion + dropout = 1")
ax.legend(frameon=False)
ax.set_title("Completion and dropout are nearly the same fact")
fig.tight_layout()
plt.show()

pair_sum = df["completion_rate"] + df["dropout_rate"]
print(
    f"completion + dropout: mean {pair_sum.mean():.3f}, "
    f"std {pair_sum.std():.3f}, min {pair_sum.min():.3f}, max {pair_sum.max():.3f}"
)
print(f"corr(completion, dropout) = {df['completion_rate'].corr(df['dropout_rate']):.3f}")


### 1.4 How teaching load and course mix look

Instructors are the modeling unit, so the batch table has to be read through that lens: how many batches each person has, and whether course identity swamps instructor differences.


In [ ]:
load = df.groupby("instructor_id").agg(
    n_batches=("batch_id", "size"),
    n_courses=("course_id", "nunique"),
)
print("Batches per instructor")
display(load["n_batches"].describe().to_frame("n_batches").T)
print("Distinct courses per instructor")
display(load["n_courses"].describe().to_frame("n_courses").T)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sns.histplot(load["n_batches"], bins=12, ax=axes[0], color="#3d5a80")
axes[0].set_title("Batches per instructor")
axes[0].set_xlabel("batches")
sns.histplot(load["n_courses"], bins=12, ax=axes[1], color="#ee6c4d")
axes[1].set_title("Distinct courses per instructor")
axes[1].set_xlabel("courses")
fig.tight_layout()
plt.show()

course_means = (
    df.groupby("course_id")["completion_rate"].mean().sort_values()
)
fig, ax = plt.subplots(figsize=(8, 5.5))
course_means.plot.barh(ax=ax, color="#98c1d9")
ax.set_xlabel("mean completion rate")
ax.set_title("Course means differ, but only inside a narrow band")
fig.tight_layout()
plt.show()

instructor_completion = df.groupby("instructor_id")["completion_rate"].mean()
print(
    f"Course-mean completion ranges from {course_means.min():.3f} to {course_means.max():.3f}"
)
print(
    f"Instructor-mean completion SD = {instructor_completion.std():.3f} "
    f"(range {instructor_completion.min():.3f} to {instructor_completion.max():.3f})"
)
print(
    "Average within-instructor SD of completion = "
    f"{df.groupby('instructor_id')['completion_rate'].std().mean():.3f}"
)


### 1.5 Early observations

- **2,000 batches, 120 instructors, 25 courses.** Batch ids are unique. There are no missing values, so there is nothing to impute. There is also no enrollment count, no batch date, and no learner-level pretest, which limits how far the score can be trusted.
- **Distributions are roughly symmetric.** Nothing needs a log transform before averaging.
- **Several metrics are censored.** Completion never falls below 0.30 (77 batches sit on that floor). Dropout never rises above 0.70. Watch time and assignment submission pile up at 1.0. Averages of censored numbers are slightly compressed: truly weak batches look a bit less weak than they were.
- **Completion and dropout are redundant.** Their correlation is about -0.95, and their sum sits near 1. Putting both in a score would count "did learners stay?" twice. Dropout is dropped from the effectiveness definition for that reason. It stays in the data dictionary as a data-quality check.
- **Other batch-level correlations are modest** (mostly 0.1 to 0.4). That does not mean instructors are interchangeable. A lot of each batch is noise around the instructor. The instructor-level view later in the notebook is where the shared pattern shows up.
- **Every instructor already has between 7 and 31 batches** (median 17) and teaches 6 to 19 courses. Nobody is a one-batch anecdote. Means for a 7-batch instructor are still noisier than means for a 30-batch instructor, so sample size has to be handled explicitly.
- **Course difficulty is a real but mild confounder.** Mean completion by course only spans about 0.56 to 0.64, while instructor means spread much more widely. Course mix can nudge a score. It does not create the ranking on its own. With no dates and no content version, a full course fixed-effect adjustment would be extra machinery for a small shift, so the primary score is not residualized. The limitation is called out again in the written answers.


## 2. Defining instructor effectiveness

### What the score is trying to measure

Effectiveness here means: **learners persist and learn more in this instructor's batches.** It is a proxy built from outcomes the platform already stores. It is not a measure of teaching quality in the classroom sense, and it is not a judgment of the person.

Three outcome pieces, each scaled to 0–1 with a min–max fit on this dataset so that a wide-range column (score improvement, roughly 6 to 40) cannot drown a 0–1 column:

| Component | Column | Weight | Why this weight |
|---|---|---|---|
| Persistence | `completion_rate` | 0.45 | Closest thing in the file to "learners got through the course," which is what an EdTech business and a learner both feel. |
| Learning gain | `avg_score_improvement` | 0.35 | Pre-to-post change is the cleanest learning signal available. It already subtracts a starting point, so it is preferred over a raw level. |
| Mastery | `avg_quiz_score` | 0.20 | Useful, but a high quiz average can mean a strong incoming cohort rather than a strong instructor. Lower weight on purpose. |

`dropout_rate` is excluded because it duplicates completion.

Engagement (watch time, assignments, forums) and feedback (rating, response rate) are **kept out of the label**. They are inputs the model is allowed to use. Mixing them into the score would make the prediction task circular, and it would also treat "learners clicked a lot" as proof of teaching effectiveness. Those behaviors have many other causes: required forum posts, video length, grade incentives, survey bias.

### Assumptions this commits us to

- A batch with no enrollment column is weighted the same as any other batch. A 20-learner batch and a 2,000-learner batch count equally. That is a data limit, not a belief that they should.
- Min–max scaling is **relative to this file**. A score of 0.8 means "high compared with these 2,000 batches," not "80% of an external standard." Frozen benchmarks would be better in production.
- Relative tertiles always label about one third of instructors Low, even if the whole pool is doing well. That fits a coaching triage. It does not fit a firing threshold.


In [ ]:
# Weights are a product decision, written down so a reviewer can change them.
OUTCOME_WEIGHTS = {
    "completion_rate": 0.45,
    "avg_score_improvement": 0.35,
    "avg_quiz_score": 0.20,
}
assert abs(sum(OUTCOME_WEIGHTS.values()) - 1) < 1e-9

outcome_scaler = MinMaxScaler()
outcome_norm = pd.DataFrame(
    outcome_scaler.fit_transform(df[list(OUTCOME_WEIGHTS)]),
    columns=list(OUTCOME_WEIGHTS),
    index=df.index,
)
df["batch_outcome_score"] = sum(
    weight * outcome_norm[col] for col, weight in OUTCOME_WEIGHTS.items()
)

print("Batch outcome score")
display(df["batch_outcome_score"].describe().to_frame("batch_outcome_score").T)
print("Min-max reference ranges used to scale the three outcome columns:")
display(
    pd.DataFrame(
        {
            "data_min": outcome_scaler.data_min_,
            "data_max": outcome_scaler.data_max_,
        },
        index=list(OUTCOME_WEIGHTS),
    )
)


## 3. Aggregating batches to instructors

### Aggregation choices

| Instructor feature | Function | Why |
|---|---|---|
| Outcome score (the label) | Unweighted mean of batch outcome scores | No enrollment weights exist. The mean uses every batch and matches the definition above. |
| Engagement and feedback predictors | Unweighted mean | Same reason. These are what the model will see. |
| Consistency | Standard deviation across batches | A coach needs to know whether a "medium" instructor is steady or a mix of excellent and poor batches. |
| Teaching load | Count of batches, count of distinct courses | Separates "teaches a lot" from "learners do well." Volume is context, not a reward. |

**Few batches vs many batches.** The minimum in this file is 7, so no instructor is dropped for being too thin, and volume is not turned into a bonus inside the score. As a sensitivity check, each instructor mean is also pulled toward the global mean with a prior worth 4 average batches:

`shrunk = (n * instructor_mean + 4 * global_mean) / (n + 4)`

That prior is stronger for someone with 7 batches (about 36% of the weight) than for someone with 31 (about 11%). If shrinkage does not change any tertile, the ranking is not an artifact of noisy small samples. The result of that check is printed below.

**Tiers.** Low / Medium / High are the bottom, middle, and top third of the shrunk score. Cuts are computed once on the full instructor pool and then frozen. They are ranks inside this group of 120, not absolute standards.


In [ ]:
SHRINKAGE_PRIOR_BATCHES = 4

instructor = df.groupby("instructor_id").agg(
    n_batches=("batch_id", "size"),
    n_courses=("course_id", "nunique"),
    outcome_mean=("batch_outcome_score", "mean"),
    outcome_std=("batch_outcome_score", "std"),
    completion_mean=("completion_rate", "mean"),
    gain_mean=("avg_score_improvement", "mean"),
    quiz_mean=("avg_quiz_score", "mean"),
    watch_mean=("avg_watch_time", "mean"),
    assignment_mean=("assignment_submission_rate", "mean"),
    forum_mean=("forum_activity_rate", "mean"),
    feedback_mean=("avg_feedback_score", "mean"),
    feedback_response_mean=("feedback_response_rate", "mean"),
    watch_std=("avg_watch_time", "std"),
    assignment_std=("assignment_submission_rate", "std"),
    forum_std=("forum_activity_rate", "std"),
    feedback_std=("avg_feedback_score", "std"),
)

global_mean = instructor["outcome_mean"].mean()
n = instructor["n_batches"]
instructor["outcome_shrunk"] = (
    n * instructor["outcome_mean"] + SHRINKAGE_PRIOR_BATCHES * global_mean
) / (n + SHRINKAGE_PRIOR_BATCHES)

# Tertiles of the raw mean vs the shrunk score: did the prior move anyone?
raw_tier = pd.qcut(instructor["outcome_mean"], q=3, labels=["Low", "Medium", "High"])
instructor["effectiveness_tier"] = pd.qcut(
    instructor["outcome_shrunk"], q=3, labels=["Low", "Medium", "High"]
)
n_flips = int((raw_tier != instructor["effectiveness_tier"]).sum())
rank_corr = instructor["outcome_mean"].corr(instructor["outcome_shrunk"], method="spearman")

tier_cuts = instructor["outcome_shrunk"].quantile([1 / 3, 2 / 3])
print(f"Global mean batch-outcome score: {global_mean:.3f}")
print(f"Spearman correlation, raw mean vs shrunk score: {rank_corr:.3f}")
print(f"Instructors whose tertile changes after shrinkage: {n_flips} of {len(instructor)}")
print(
    f"Frozen cuts on the shrunk score: "
    f"Low < {tier_cuts.iloc[0]:.3f} <= Medium < {tier_cuts.iloc[1]:.3f} <= High"
)
display(instructor["effectiveness_tier"].value_counts().sort_index().to_frame("instructors"))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(instructor["outcome_shrunk"], bins=18, ax=axes[0], color="#3d5a80")
for cut in tier_cuts:
    axes[0].axvline(cut, color="#c44536", ls="--", lw=1)
axes[0].set_title("Instructor effectiveness score")
axes[0].set_xlabel("shrunk outcome score")
sns.boxplot(
    data=instructor.reset_index(),
    x="effectiveness_tier",
    y="outcome_std",
    order=["Low", "Medium", "High"],
    ax=axes[1],
    color="#98c1d9",
)
axes[1].set_title("Within-instructor spread of the batch score")
axes[1].set_xlabel("")
axes[1].set_ylabel("SD of batch outcome score")
fig.tight_layout()
plt.show()

# Batch-level correlations looked modest. Instructor means line up much more tightly
# because batch noise averages out. This is why the model has something to learn.
predictor_means = [
    "watch_mean",
    "assignment_mean",
    "forum_mean",
    "feedback_mean",
    "feedback_response_mean",
    "n_batches",
    "n_courses",
    "outcome_std",
]
print("Correlation of the shrunk outcome score with instructor-level predictors")
display(
    instructor[predictor_means]
    .corrwith(instructor["outcome_shrunk"])
    .sort_values(ascending=False)
    .to_frame("corr_with_outcome_score")
)


Shrinkage does not change a single tertile in this file. Every instructor already has at least 7 batches, so a prior of 4 batches is not strong enough to move anyone across a tertile boundary. The check is still worth keeping: the same definition applied to a future cohort of brand-new instructors would need it. The official label below is the shrunk score, which in this sample is effectively the same ranking as the plain mean.

The correlation table is the substantive result of the aggregation. At batch level, feedback and engagement only weakly tracked completion. At instructor level, feedback response, feedback rating, forum activity, watch time, and assignment submission all correlate with the outcome score at roughly 0.7 to 0.85. Teaching load does not (correlation near 0). Averaging batches revealed a stable instructor-level pattern that any one batch hides. That pattern is also a warning: the predictors and the label move together so closely that part of the signal may be "this cohort was easy," not "this instructor caused the outcome."


## 4. Model

### Target and features

**Target:** `effectiveness_tier` (Low / Medium / High), three balanced classes of 40 by construction.

**Features the model may see:**

- Means: watch time, assignment submission, forum activity, feedback rating, feedback response rate
- Consistency: standard deviation of watch time, assignment submission, forum activity, and feedback rating
- Context: number of batches, number of distinct courses

**Features the model may not see:** completion, dropout, score improvement, quiz score, and the outcome score itself. Those are the label, or a near-copy of the label.

### Algorithm

Multinomial logistic regression, with inputs standardized inside the pipeline so coefficients are comparable. Chosen because the sample is 120 instructors, the question is "which signals point which way," and a linear model is easy to audit.

A random forest (depth-limited) is a second view, in case the tiers depend on interactions the linear model misses. If the two agree, the extra complexity is not buying much.

### Validation

Five-fold stratified cross-validation, shuffle, `random_state=42`. With 120 rows, a single 70/30 split would judge the model on about 36 instructors and bounce around. Out-of-fold predictions from the five folds give one prediction per instructor without training on that instructor.

**Metrics**

- **Macro F1** is the headline. The classes are equal size, and macro F1 refuses to let a model look good by only getting the easy class right.
- **Accuracy** is reported next to a chance baseline of 1/3 (the classes are balanced, so a coin-flip among three tiers lands there). A stratified dummy classifier is the empirical baseline.
- **Quadratic weighted kappa** treats the tiers as ordered. Calling a High instructor Low is a worse mistake than calling them Medium. Ordinary accuracy treats those two errors as identical.
- Per-class precision and recall, plus the confusion matrix, show *where* the errors sit. Medium is expected to be the hard class because both of its borders are close.

**Operating point.** Default predictions use the highest probability. For a coaching product, missing a Low instructor (false negative) is more costly than a supportive check-in with someone who is actually Medium. A second operating point flags Low when `P(Low) >= 0.35`, and the precision/recall change is reported. That is a product choice, not a better accuracy number.


In [ ]:
FEATURE_COLS = [
    "watch_mean",
    "assignment_mean",
    "forum_mean",
    "feedback_mean",
    "feedback_response_mean",
    "watch_std",
    "assignment_std",
    "forum_std",
    "feedback_std",
    "n_batches",
    "n_courses",
]
FEATURE_LABELS = {
    "watch_mean": "Watch time (mean)",
    "assignment_mean": "Assignment submission (mean)",
    "forum_mean": "Forum activity (mean)",
    "feedback_mean": "Feedback rating (mean)",
    "feedback_response_mean": "Feedback response rate (mean)",
    "watch_std": "Watch time (SD)",
    "assignment_std": "Assignment submission (SD)",
    "forum_std": "Forum activity (SD)",
    "feedback_std": "Feedback rating (SD)",
    "n_batches": "Number of batches",
    "n_courses": "Number of courses",
}

X = instructor[FEATURE_COLS]
y = instructor["effectiveness_tier"]
class_order = ["Low", "Medium", "High"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

logreg = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        (
            "model",
            LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        ),
    ]
)
forest = Pipeline(
    steps=[
        (
            "model",
            # No scaling: trees do not use feature magnitude.
            # Shallow trees and a leaf minimum keep ~96 training rows from being memorized.
            RandomForestClassifier(
                n_estimators=300,
                max_depth=4,
                min_samples_leaf=3,
                random_state=RANDOM_STATE,
            ),
        )
    ]
)
dummy = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)


def evaluate(name, estimator):
    pred = cross_val_predict(estimator, X, y, cv=cv)
    acc = accuracy_score(y, pred)
    macro = f1_score(y, pred, average="macro", labels=class_order)
    kappa = cohen_kappa_score(y, pred, labels=class_order, weights="quadratic")
    return {
        "model": name,
        "accuracy": acc,
        "macro_f1": macro,
        "quadratic_kappa": kappa,
        "predictions": pred,
    }


results = [
    evaluate("stratified dummy", dummy),
    evaluate("logistic regression", logreg),
    evaluate("random forest", forest),
]
summary = pd.DataFrame(results).drop(columns="predictions")
display(summary)

oof = results[1]["predictions"]  # logistic regression out-of-fold labels
print("Logistic regression, out-of-fold classification report")
print(classification_report(y, oof, labels=class_order, digits=3))

cm = confusion_matrix(y, oof, labels=class_order)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_order,
    yticklabels=class_order,
    ax=ax,
)
ax.set_xlabel("Predicted tier")
ax.set_ylabel("Actual tier")
ax.set_title("Logistic regression, out-of-fold")
fig.tight_layout()
plt.show()

# Far errors (Low predicted High, or High predicted Low) vs adjacent errors.
low_as_high = int(cm[class_order.index("Low"), class_order.index("High")])
high_as_low = int(cm[class_order.index("High"), class_order.index("Low")])
print(f"Low predicted as High: {low_as_high}")
print(f"High predicted as Low: {high_as_low}")


## 5. Evaluation

Read the table above against three reference points, not against 100%:

- A stratified dummy lands near **0.33 accuracy**, because the tiers are balanced by construction. Anything near that number has not learned the instructors.
- **Macro F1 around 0.73** for both logistic regression and the forest means the signal is real and mostly linear. The forest does not unlock a meaningfully better decision, so the logistic model is the one to interpret.
- **Quadratic weighted kappa near 0.60** says the remaining mistakes are mostly neighbors. Swapping Low and High is rare. Medium is where precision and recall both sit near 0.60: those instructors live on the cut points, and a small change in one batch can move them.

**Precision vs recall for the Low tier.** Under the default rule (highest probability), Low recall is about 0.73 and Low precision is about 0.71. Roughly one in four instructors who land in the bottom third would not be flagged, and roughly one in four flags would be a Medium instructor. There are essentially no Low-as-High misses.

**Class imbalance.** There is none in the label, because tertiles force 40/40/40. That is convenient for training and easy to misuse. In a later cohort, if cuts stay frozen at the numbers printed above, the classes will not stay equal, and accuracy will stop being comparable to the 0.73 reported here. Macro F1 and the confusion matrix should travel with the model; accuracy alone should not.

**What this number is not.** It is not evidence that the model measures teaching skill. It is evidence that instructors whose learners watch, submit, post, and answer surveys also tend to sit in a higher outcome third. Section 7 says where that reasoning breaks.


In [ ]:
# Coaching operating point: flag Low a bit earlier than the default argmax rule.
oof_proba = cross_val_predict(logreg, X, y, cv=cv, method="predict_proba")
# Column order follows the classifier's class_ order, recovered from a fitted clone.
fitted = logreg.fit(X, y)
proba_columns = list(fitted.named_steps["model"].classes_)
low_index = proba_columns.index("Low")
p_low = oof_proba[:, low_index]
actually_low = (y == "Low").to_numpy()

rows = []
for threshold in (0.50, 0.35, 0.25):
    flagged = p_low >= threshold
    tp = int((flagged & actually_low).sum())
    fp = int((flagged & ~actually_low).sum())
    fn = int((~flagged & actually_low).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    rows.append(
        {
            "rule": f"flag Low if P(Low) >= {threshold:.2f}",
            "flagged": int(flagged.sum()),
            "precision_on_Low": precision,
            "recall_on_Low": recall,
        }
    )
display(pd.DataFrame(rows))


Lowering the flag from 0.50 to 0.35 catches more of the bottom third and also pulls in more Medium instructors. That trade is reasonable for an optional coaching queue. It is not reasonable for a performance rating, where a false positive has a cost to the person, not just a cost in staff time.


### A deliberately leaky model, for contrast

The same logistic setup, but fed the three outcome means that *define* the score. This is what "predict the tier from the columns inside the tier" looks like. It is included so a high accuracy number is not confused with a useful model.


In [ ]:
leaky_features = ["completion_mean", "gain_mean", "quiz_mean"]
X_leaky = instructor[leaky_features]
leaky_model = Pipeline(
    [
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]
)
leaky_pred = cross_val_predict(leaky_model, X_leaky, y, cv=cv)
print(
    f"Leaky model accuracy {accuracy_score(y, leaky_pred):.3f}, "
    f"macro F1 {f1_score(y, leaky_pred, average='macro'):.3f}"
)
print(
    "This model is not used for any conclusion. "
    "It recovers a label that was computed from these same columns."
)


## 6. What drives the prediction

Permutation importance is the average drop in out-of-fold macro F1 when a column is shuffled inside each test fold (30 repeats). A larger drop means the model was relying on that column. Coefficients come from one logistic regression fit on all 120 instructors, after standardization, so they are in "per standard deviation" units. They are for direction and conversation. The cross-validated metrics above are the evaluation.


In [ ]:
fold_importances = []
for train_idx, test_idx in cv.split(X, y):
    fold_model = logreg.fit(X.iloc[train_idx], y.iloc[train_idx])
    result = permutation_importance(
        fold_model,
        X.iloc[test_idx],
        y.iloc[test_idx],
        n_repeats=30,
        random_state=RANDOM_STATE,
        scoring="f1_macro",
    )
    fold_importances.append(result.importances_mean)

importance = (
    pd.Series(np.mean(fold_importances, axis=0), index=FEATURE_COLS)
    .sort_values(ascending=True)
)
fig, ax = plt.subplots(figsize=(8, 4.8))
importance.plot.barh(ax=ax, color="#3d5a80")
ax.set_xlabel("Mean drop in macro F1 when shuffled")
ax.set_title("What the logistic model actually uses")
ax.set_yticklabels([FEATURE_LABELS[c] for c in importance.index])
fig.tight_layout()
plt.show()
display(importance.sort_values(ascending=False).to_frame("permutation_importance"))

fitted = logreg.fit(X, y)
coef = pd.DataFrame(
    fitted.named_steps["model"].coef_,
    index=fitted.named_steps["model"].classes_,
    columns=FEATURE_COLS,
).T
coef.index = [FEATURE_LABELS[c] for c in coef.index]
print("Standardized logistic coefficients (positive pushes toward that tier)")
display(coef.reindex(columns=["Low", "Medium", "High"]))

# Profile in original units, so the coefficients have a concrete scale.
profile_cols = [
    "watch_mean",
    "assignment_mean",
    "forum_mean",
    "feedback_mean",
    "feedback_response_mean",
    "n_batches",
]
profile = instructor.groupby("effectiveness_tier", observed=True)[profile_cols].mean().T
profile.index = [FEATURE_LABELS[c] for c in profile.index]
profile = profile.reindex(columns=class_order)
print("Average predictor values by tier")
display(profile)


## 7. Reading the result in plain language

Instructors in the top third are not the ones who teach the most courses. Batch count and course count barely move the model. They are the instructors whose learners, across batches, do three things together: they finish and gain more (that is the label), and they also watch more of the material, submit more assignments, post more, and — especially — both *rate* the experience higher and *bother to send* that rating.

The two feedback columns carry the most weight, then forum activity and assignment submission, then watch time. Less consistent watch time points away from the top third: instructors whose batches swing from "everyone watches" to "almost nobody watches" are less often in the high-outcome group. A stable pattern beats a flashy batch.

**How a product could use this, without turning it into a score that follows people around**

- A coaching queue, not a leaderboard. The Low flag at `P(Low) >= 0.35` is a reason for a human to look at the batches, not a reason to publish a rank.
- Pair the flag with the raw ingredients: which of completion, learning gain, or quiz mastery is the weak one, and whether feedback response is low because learners are disengaged or because the survey is easy to ignore.
- Watch consistency. An instructor with one poor batch and otherwise solid batches is a different conversation from an instructor whose every batch is soft.
- Do not pay, promote, or remove people on this tier. Section 8 explains why the model cannot carry that decision.

The leaky model at ~0.97 accuracy is the cautionary result. A dashboard that "predicts effectiveness" from completion and quiz score is just renaming the score. The honest model, around 0.73 macro F1, is the one that asks a real question: do engagement and feedback anticipate outcome tiers the label did not already include?


## 8. Mandatory analysis questions

### 1. Which features most influenced instructor effectiveness, and why?

On the model that is not allowed to see the outcome columns, permutation importance ranks:

1. **Feedback response rate** (about 0.12 macro-F1 drop when shuffled)
2. **Feedback rating** (about 0.09)
3. **Watch-time consistency**, then **assignment submission** and **forum activity** (about 0.05 to 0.07)
4. Watch-time level, feedback consistency, and both teaching-load counts, which matter little
5. Forum-activity consistency, which the model does not use in a stable way (the shuffle drop is near zero or slightly negative)

Why these, in this dataset: once batches are averaged, feedback and engagement line up with the outcome score at correlations of roughly 0.7 to 0.85. Learners who stay, learn, and score well are also the learners who watch, submit, post, and answer the survey. Response rate outranking the rating itself is informative. It says "did a large share of the cohort bother to respond?" carries at least as much tier information as the stars they gave. That can mean a genuinely engaged cohort. It can also mean the survey only gets answered when the batch already went well. Both readings are consistent with the coefficients: higher response rate and higher rating push hard toward High and away from Low.

Teaching volume is not an effectiveness feature here. Number of batches and number of courses have correlations near zero with the score and almost no permutation importance. An instructor is not more effective because they were assigned more batches.

### 2. Which variables could be misleading or confounded?

- **Dropout vs completion.** Correlation about -0.95. Using both would pretend there are two retention signals. Dropout was left out of the score for this reason.
- **Quiz score vs incoming ability.** The file has a gain column and a level column. The level can be high because the cohort arrived strong. Gain is the better learning measure, which is why quiz weight is only 0.20, but quiz is still partly a cohort feature.
- **Feedback rating vs who answers.** A 4.8 from 30% of learners is not the same fact as a 4.8 from 90%. Response rate is in the model as its own feature, but the rating is not reweighted by response rate, so a small enthusiastic group can still pull an instructor up.
- **Watch time, forums, and assignments vs course design.** Required posts, long videos, and graded assignments change these numbers without any change in teaching. The model will treat a course-design effect as an instructor effect.
- **Course difficulty.** Course-mean completion only spans about 0.56 to 0.64, so this file's catalog is fairly even. A harder course added later would make its instructors look less effective under a score that is not course-adjusted.
- **Censoring.** Completion is floored at 0.30 and dropout is capped at 0.70. The worst batches are squeezed toward the middle, so the bottom of the ranking is flatter than reality.
- **No enrollment weights.** A tiny batch and a huge batch count the same. One unusual small cohort can move a 7-batch instructor more than it should.
- **The label is ours.** Any model of this tier is partly a model of the weights 0.45 / 0.35 / 0.20. Different weights would move people near the cuts.

### 3. How could this model fail in real-world usage?

- **Relative tiers always produce a bottom third.** If the whole instructor pool improves, someone is still labeled Low. Used as a target, that punishes people for the existence of a ranking.
- **Cold start.** The shrinkage check did nothing here because nobody has fewer than 7 batches. A new instructor after one batch would get a shrunk score and a very uncertain tier. The model has not been shown that case.
- **Cohort effects.** If a batch of already-strong learners both finishes and fills in the survey, the model credits the instructor. The file cannot separate learner intake from teaching.
- **Gaming.** If coaches announce that forum posts, watch time, and survey response raise the tier, instructors can require clicks. Those features would keep "predicting" the tier while the outcome link rots. This is the usual way a proxy gets destroyed once it becomes a target.
- **Distribution shift.** New courses, a new survey, a switch from optional to mandatory assignments, or a different learner population would move the inputs. The frozen cuts (printed in section 3) would then mean something else, and the 0.73 macro F1 would not travel with them.
- **Boundary errors.** Medium is the noisy class. A product that treats Medium and Low as sharply different will thrash people whose true score sits on the cut.
- **High-stakes individual decisions.** 120 instructors and a hand-built label are enough for a triage hypothesis. They are not enough to justify a recorded performance rating.

### 4. What additional data would you want to improve this analysis?

In rough order of how much they would change the conclusion:

- **Enrollment per batch**, so means can be weighted by learners rather than by batches.
- **A real pretest at learner level**, separate from the batch-average gain, to hold incoming ability constant.
- **Batch timing and content version**, so a bad term or a bad content release is not stored as an instructor trait, and so a trend (improving vs declining) can be seen.
- **Course difficulty or a course identifier inside the model**, once the catalog is less uniform than this one.
- **Who else is in the room:** teaching assistants, support response time, live sessions versus fully recorded.
- **Survey text and a non-response study**, so the rating is not doing extra work it has not earned.
- **A downstream outcome** the score does not already contain: the next course started, a project reviewed by a human, or a job-relevant task. Without that, "effectiveness" remains a re-expression of completion, gain, and quizzes.
- **Fairness slices** (learner segment, language, timezone) to see whether the tier gap is an instructor gap or a mix gap.

### 5. Should this model be used for instructor performance evaluation? Why or why not?

**No, not as an evaluation of the person, and not as an input to pay, promotion, or exit.**

The tier is a bucket we defined from three outcome averages, then predicted from behaviors that are entangled with those outcomes and with course design. Cross-validated macro F1 around 0.73, with almost all mistakes on the Medium border, is a decent description of this file. It is not a validated measure of teaching quality. There is no learner-level prior, no class size, no time dimension, and no outcome that the label did not already include. Feedback can be biased by who responds. Relative tertiles guarantee a Low group even when everyone is fine. Once instructors know the features, they can move the prediction without moving learning.

**Yes, as a private triage signal for a human reviewer**, with the raw batch metrics shown beside the flag and with an explicit note that the flag is a hypothesis. The useful product sentence is: "these instructors' recent batches are in the lower third of persistence and learning gain, and their engagement and feedback look the same way — someone should look." The sentence that is not supported is: "this model says this instructor is ineffective."


## 9. What would be revisited first

If the weights changed, the people nearest the two cuts would move tiers, and the reported F1 would change with them. That sensitivity is a reason to publish the weights next to any dashboard number. The next analytical step, given more data rather than more model, would be enrollment-weighted means and a course adjustment. A more complex classifier is not the bottleneck. The forest already matched the logistic model, which says the remaining errors are overlap in the data, not a missing interaction.
